In [15]:
import os
import pandas as pd
import numpy as np
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys
from rdkit.Chem.AllChem import GetMorganGenerator, GetRDKitFPGenerator
from rdkit import DataStructs
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [16]:
# Load dataset
path = r"C:\Users\d0m1n\Desktop\VSCode\AI.ML.ENGR\AI.ML HW4\Lipophilicity.csv"
df = pd.read_csv(path)
df.head()

,CMPD_CHEMBLID,exp,smiles
0,CHEMBL596271,3.54,Cn1c(CN2CCN(CC2)c3ccc(Cl)cc3)nc4ccccc14
1,CHEMBL1951080,-1.18,COc1cc(OC)c(cc1NC(=O)CSCC(=O)O)S(=O)(=O)N2C(C)...
2,CHEMBL1771,3.69,COC(=O)[C@@H](N1CCc2sccc2C1)c3ccccc3Cl
3,CHEMBL234951,3.37,OC[C@H](O)CN1C(=O)C(Cc2ccccc12)NC(=O)c3cc4cc(C...
4,CHEMBL565079,3.10,Cc1cccc(C[C@H](NC(=O)c2cc(nn2C)C(C)(C)C)C(=O)N...


In [17]:
# Label Columns
SMILES_COL = 'smiles'
TARGET_COL = 'exp'

In [18]:
# Convert SMILES to Molecular Fingerprints
df['Mol'] = df[SMILES_COL].apply(Chem.MolFromSmiles)

In [19]:
# Clean Data
df = df.dropna(subset=['Mol', TARGET_COL])
y = df[TARGET_COL].values

In [20]:
# Generate Morgan Fingerprint Generator
def morgan_fp(mol, radius=2, fpSize=2048):
    if mol is None:
        return np.zeros((fpSize,), dtype=np.int8)
    generator = AllChem.GetMorganGenerator(radius=radius, fpSize=fpSize)
    fp = generator.GetFingerprint(mol)
    arr = np.zeros((fpSize,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

In [21]:
# Generate MACCS Keys
def maccs_fp(mol):
    if mol is None:
        return np.zeros((167,), dtype=np.int8)
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((167,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

In [22]:
# Generate Features
X_morgan = np.vstack(df['Mol'].apply(morgan_fp))
X_maccs = np.vstack(df['Mol'].apply(maccs_fp))

In [23]:
# Split Dataset
X_morgan_train, X_morgan_test, y_train, y_test = train_test_split(
    X_morgan, y, test_size=0.2, random_state=42
)
X_maccs_train, X_maccs_test, _, _ = train_test_split(
    X_maccs, y, test_size=0.2, random_state=42
)

In [24]:
# Scale Features
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

In [25]:
# MLP Regressor Parameters
mlp_params = {
    'hidden_layer_sizes': (100, 50),
    'activation': 'relu',
    'solver': 'adam',
    'max_iter': 500,
    'random_state': 42
}

In [26]:
# Morgan Fingerprint Model
morgan_model = MLPRegressor(**mlp_params)
morgan_model.fit(X_morgan_train, y_train)

# Generate Predictions
morgan_predictions_scaled = morgan_model.predict(X_morgan_test)
morgan_predictions_scaled = morgan_predictions_scaled.reshape(-1, 1)
morgan_predictions = y_scaler.inverse_transform(morgan_predictions_scaled).ravel()

# Evaluate Morgan Model
morgan_rmse = np.sqrt(mean_squared_error(y_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel(), morgan_predictions))
morgan_r2 = r2_score(y_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel(), morgan_predictions)

print(f"Morgan Fingerprint Model - RMSE: {morgan_rmse:.4f}, R2: {morgan_r2:.4f}")

Morgan Fingerprint Model - RMSE: 0.7492, R2: 0.6201


In [27]:
# MACCS Keys Model
maccs_model = MLPRegressor(**mlp_params)
maccs_model.fit(X_maccs_train, y_train)

# Generate Predictions
maccs_predictions_scaled = maccs_model.predict(X_maccs_test)
maccs_predictions_scaled = maccs_predictions_scaled.reshape(-1, 1)
maccs_predictions = y_scaler.inverse_transform(maccs_predictions_scaled).ravel()

# Evaluate MACCS Model
maccs_rmse = np.sqrt(mean_squared_error(y_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel(), maccs_predictions))
maccs_r2 = r2_score(y_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel(), maccs_predictions)
print(f"MACCS Keys Model - RMSE: {maccs_rmse:.4f}, R2: {maccs_r2:.4f}")

MACCS Keys Model - RMSE: 0.9474, R2: 0.3925
